## PydanticAI 프롬프트 엔지니어링 실습

**프롬프트 엔지니어링이란?**
- AI 모델로부터 원하는 결과를 얻기 위해 **효과적인 질문/명령**을 설계하는 기술
- 같은 질문이라도 **프롬프트를 어떻게 작성하느냐**에 따라 응답의 품질이 크게 달라짐
- 데이터 분석가에게 필수적인 스킬: 데이터 해석, 인사이트 도출, 보고서 작성 등에 활용

**이 노트북에서는** PydanticAI를 활용하여 프롬프트 엔지니어링 기법을 실습합니다.

**프롬프트 엔지니어링 참고 자료:**
- [Gemini Prompt Engineering](https://ai.google.dev/gemini-api/docs/prompting-strategies?hl=ko)
- [OpenAI Prompt Engineering](https://platform.openai.com/docs/guides/prompt-engineering)
- [Anthropic Prompt Engineering](https://docs.anthropic.com/claude/docs/prompt-engineering)

In [ ]:
# 필요한 패키지를 한 번에 설치합니다
# !pip install -q pydantic-ai-slim[google] python-dotenv pydantic

# uv 권장! uv 환경에서는 위 pip 대신 터미널에서 uv sync 명령어로 설치하세요:
# uv sync

In [ ]:
# ========================================
# 필요한 라이브러리를 불러옵니다
# ========================================

import os
from pprint import pprint
from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModelSettings

# .env 파일에서 API 키 로드
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-3.1-flash-lite-preview')
model_id = f'google-gla:{gemini_model}'

# API 키 유효성 검사
api_key_valid = api_key and 'YOUR_API_KEY' not in api_key
print(f"API 키 설정 확인: {'✓' if api_key_valid else '✗'}")
if not api_key_valid:
    print("  .env 파일에서 GEMINI_API_KEY를 실제 API 키로 설정해주세요!")
print(f"모델 확인: {model_id}")

# 기본 설정
default_settings = GoogleModelSettings(temperature=0.7, max_tokens=1000, top_p=0.95)

### 1. 명확한 지침 제공 (Clear Instructions)

**핵심 원칙**
- 중요한 세부사항, 문맥, 제약조건을 명시
- 모호한 표현 대신 구체적인 요구사항 전달
- 출력 형식과 길이 지정

PydanticAI에서는 **같은 Agent**에 다른 프롬프트를 넣어 비교할 수 있습니다.

In [ ]:
print("=" * 80)
print("실습 1: 명확성의 중요성 - 데이터 분석 시나리오")
print("=" * 80)

# 기본 Agent 생성
basic_agent = Agent(model_id)

# [X] 나쁜 예시: 모호한 프롬프트
bad_prompt = "데이터 분석에 대해 설명해줘"

print(f"\n[X] 나쁜 프롬프트:\n{bad_prompt}\n")
print("AI 응답:")

result_bad = await basic_agent.run(
    bad_prompt,
    model_settings=GoogleModelSettings(temperature=0.7, max_tokens=100)
)
pprint(result_bad.output)

print("\n" + "-" * 80 + "\n")

# [O] 좋은 예시: 명확하고 구체적인 프롬프트
good_prompt = """다음 조건에 맞춰 A/B 테스트 결과를 분석하는 방법을 설명해줘:

**상황:**
- 이커머스 사이트의 '구매하기' 버튼 색상 변경 테스트 (빨강 vs 파랑)
- A그룹(빨강): 1000명, 전환율 5.2%
- B그룹(파랑): 1000명, 전환율 6.8%

**요구사항:**
1. 통계적 유의성 판단 방법 (카이제곱 검정 기준)
2. 비즈니스 의사결정을 위한 해석 (2-3문장)
3. 주의사항 1가지

**제약:**
- 전체 5문장 이내
- 전문용어 사용 시 괄호 안에 쉬운 설명 추가
"""

print(f"[O] 좋은 프롬프트:\n{good_prompt}")
print("AI 응답:")

result_good = await basic_agent.run(
    good_prompt,
    model_settings=GoogleModelSettings(temperature=0.7, max_tokens=500)
)
pprint(result_good.output)

### 2. 페르소나 기반 역할 부여 (Role Prompting)

**핵심 개념**
- AI에게 특정 전문가 역할을 부여하여 답변의 품질과 관점 조정
- 데이터 분석가, 통계학자, 비즈니스 컨설턴트 등 역할 지정

**PydanticAI에서의 역할 부여:**

`Agent` 생성 시 `system_prompt`로 역할을 고정합니다.
**역할별로 Agent를 분리**할 수 있어 코드가 더 명확해집니다.

```python
analyst_agent = Agent(model, system_prompt="당신은 데이터 분석가입니다")
consultant_agent = Agent(model, system_prompt="당신은 비즈니스 컨설턴트입니다")
```

In [ ]:
print("=" * 80)
print("실습 2: 역할 부여 - 같은 데이터, 다른 관점")
print("=" * 80)

# 시나리오: 고객 이탈률 데이터 분석
data_context = """
**고객 이탈 데이터:**
- 전체 고객: 10,000명
- 이탈 고객: 1,200명 (12%)
- 주요 이탈 사유: 가격(45%), 경쟁사 이동(30%), 서비스 불만(25%)
- 평균 구독 기간: 이탈 고객 8개월 vs 유지 고객 24개월
"""

user_prompt = f"""
다음 고객 이탈 데이터를 분석하고 인사이트를 도출하세요.

{data_context}
"""

# 역할별 Agent 생성 — PydanticAI는 역할별로 Agent를 분리할 수 있음
analyst_agent = Agent(
    model_id,
    system_prompt="""당신은 3년 차 데이터 분석가입니다.

다음 형식으로 분석 결과를 제공하세요:
1. 핵심 지표 3가지 해석
2. 데이터 기반 개선 방안 2가지
3. 추가 분석이 필요한 항목 1가지

6문장 이내로 작성하고, 숫자와 비율을 활용하여 구체적으로 설명하세요."""
)

print("\n[역할: 데이터 분석가]")
print("AI 응답:")

result = await analyst_agent.run(user_prompt, model_settings=default_settings)
print(result.output)

### 3. 구조화된 구분자 활용 (Delimiter Usage)

**핵심 개념**
- `<데이터>`, `<출력형식>` 등 구분자로 프롬프트의 각 영역을 명확히 분리
- AI가 입력 데이터와 출력 형식을 혼동하지 않도록 함
- 복잡한 프롬프트일수록 구분자의 효과가 큼

In [ ]:
print("=" * 80)
print("실습 3: 구분자를 활용한 데이터 요약")
print("=" * 80)

delimiter_prompt = """
다음 <데이터>를 분석하여 요약 리포트를 작성하세요.

<데이터>
2024년 1분기 매출 데이터 분석 결과입니다. 
총 매출은 전년 대비 23% 증가한 150억원을 기록했습니다.
제품별로는 A제품이 60억(40%), B제품이 45억(30%), C제품이 45억(30%)을 차지했습니다.
지역별로는 서울 70억(47%), 경기 50억(33%), 기타 30억(20%)으로 수도권이 80%를 차지합니다.
온라인 채널 비중이 전년 45%에서 55%로 증가하며 디지털 전환이 가속화되고 있습니다.
고객 만족도는 평균 4.2/5.0점으로 전년 대비 0.3점 상승했습니다.
</데이터>

<출력형식>
## 핵심 지표
- 항목: 수치 (해석)

## 주요 인사이트
1. 인사이트 1
2. 인사이트 2
3. 인사이트 3

## 주의점
- 주의사항
</출력형식>

제약: 전체 8문장 이내, 각 인사이트는 1-2문장
"""

print("AI 응답:")

result = await basic_agent.run(delimiter_prompt, model_settings=default_settings)
print(result.output)

### 4. Few-Shot Learning (예시 기반 학습)

**핵심 개념**
- 원하는 출력 형식의 예시를 2-3개 제공
- AI가 패턴을 학습하여 동일한 형식으로 응답
- 일관된 보고서나 문서 작성에 매우 효과적

PydanticAI에서 Few-Shot을 구현하는 방법은 두 가지입니다:

| 방법 | 설명 | 적합한 상황 |
|------|------|------------|
| **system_prompt에 예시 포함** | Agent의 기본 동작으로 예시 패턴 학습 | 항상 같은 형식으로 응답해야 할 때 |
| **user prompt에 예시 포함** | 특정 요청에만 예시 제공 | 일회성 형식 지정 |

In [ ]:
print("=" * 80)
print("실습 4: 감정 분석 인사이트 카드 생성")
print("=" * 80)

# system_prompt에 역할과 출력 형식을 고정
insight_agent = Agent(
    model_id,
    system_prompt="""당신은 소셜 미디어 감정 분석 전문가입니다.

다음 형식으로 감정 분석 결과를 '인사이트 카드' 형태로 작성해주세요:

인사이트: [핵심 발견 사항을 한 문장으로]
데이터: [구체적인 수치와 변화]
해석: [감정 변화의 원인 또는 의미]
액션: [실행 가능한 대응 방안 2가지]

간결하고 명확하게 작성하세요."""
)

# user prompt에 Few-Shot 예시 포함
few_shot_prompt = """[예시 1]
인사이트: 신제품 출시 후 긍정 감정 급증
데이터: 긍정 댓글 비율 45% -> 72%로 증가 (출시 2주 후), "만족", "추천" 키워드 3배 증가
해석: 신제품에 대한 고객 기대가 실제 경험으로 충족되며 긍정적 입소문 확산
액션: 긍정 리뷰 활용한 SNS 마케팅 강화, 만족 고객 대상 리워드 프로그램 런칭

[예시 2]
인사이트: 배송 관련 부정 감정 집중 발견
데이터: 부정 댓글 중 62%가 배송 이슈, "늦음", "지연" 키워드가 전월 대비 40% 증가
해석: 물류 프로세스 문제로 고객 불만 누적, 재구매 의향 저하 우려
액션: 배송 프로세스 긴급 점검 및 개선, 지연 고객 대상 보상 정책 수립

[새로운 데이터 분석 요청]
데이터: 영화 리뷰 댓글 1,000개 분석 결과
- 긍정: 680건 (68%), 주요 키워드: "감동", "명작", "최고의 연기"
- 부정: 220건 (22%), 주요 키워드: "지루함", "예측 가능", "과대평가"
- 중립: 100건 (10%)
- 전작 대비 긍정 비율 15%p 상승
"""

print("분석 대상: 영화 리뷰 감정 분석 결과 (1,000개 댓글)")
print("\nAI 응답:")

result = await insight_agent.run(
    few_shot_prompt,
    model_settings=GoogleModelSettings(temperature=0.7, max_tokens=400)
)
print(result.output)

### 5. Chain-of-Thought (단계별 사고 유도)

**핵심 개념**
- 복잡한 분석 문제를 단계별로 분해
- AI가 중간 추론 과정을 명시하도록 유도
- 논리적 오류 감소, 투명한 의사결정 지원

In [ ]:
print("=" * 80)
print("실습 5: 복잡한 비즈니스 문제 해결")
print("=" * 80)

cot_prompt = """다음 비즈니스 문제를 단계별로 분석하세요.

**상황:**
온라인 쇼핑몰의 월 매출이 3개월 연속 감소하고 있습니다.
- 1월: 10억 -> 2월: 9.5억 (-5%) -> 3월: 8.8억 (-7.4%)
- 방문자 수는 유지 (월 50만명)
- 전환율: 4% -> 3.5% -> 3.2% 
- 평균 객단가: 50,000원 -> 54,000원 -> 55,000원
- 신규 고객 비율: 60% -> 55% -> 48%

**분석 프로세스:**

STEP 1: 데이터 해석
- 각 지표의 변화 추이와 의미를 분석하세요.

STEP 2: 근본 원인 가설
- 매출 감소의 가능한 원인 3가지를 제시하고, 각각을 데이터로 뒷받침하세요.

STEP 3: 추가 확인 필요 사항
- 원인을 정확히 파악하기 위해 추가로 확인해야 할 데이터 2가지

STEP 4: 해결 방안
- 가장 가능성 높은 원인에 대한 해결책 2가지 (우선순위 순)

각 STEP을 명확히 구분하여 작성하고, 전체 10문장 이내로 작성하세요.
"""

# 시니어 분석가 역할의 Agent
senior_agent = Agent(
    model_id,
    system_prompt="당신은 시니어 데이터 분석가입니다. 논리적이고 체계적으로 분석하세요."
)

print("AI 응답:")

result = await senior_agent.run(cot_prompt, model_settings=default_settings)
print(result.output)

### 7. 프롬프트 체이닝 (Prompt Chaining)

**핵심 개념**
- 복잡한 분석을 여러 단계로 나누어 순차적으로 실행
- 각 단계의 출력이 다음 단계의 입력이 됨
- 정확도 향상 및 디버깅 용이

**PydanticAI에서의 체이닝:**

`message_history`를 활용하여 이전 단계의 대화 맥락을 자동으로 전달합니다.

```python
# history 리스트에 누적하는 방식
history = []

result1 = await agent.run('STEP 1 분석', message_history=history)
history.extend(result1.new_messages())

result2 = await agent.run('STEP 2 분석', message_history=history)
history.extend(result2.new_messages())
```

In [ ]:
print("=" * 80)
print("실습 7: 다단계 고객 세그먼트 분석 (프롬프트 체이닝)")
print("=" * 80)

# 원본 데이터
customer_data = """
고객ID, 연령, 구매횟수, 총구매액, 최근구매일
C001, 28, 15, 450000, 2024-03-10
C002, 45, 3, 120000, 2024-01-05
C003, 35, 25, 1200000, 2024-03-15
C004, 52, 8, 340000, 2024-02-20
C005, 31, 18, 680000, 2024-03-12
"""

# 분석용 Agent
chain_agent = Agent(
    model_id,
    system_prompt="당신은 CRM 데이터 분석 전문가입니다. 간결하고 정확하게 분석하세요."
)

print(f"원본 데이터:{customer_data}")

# --- STEP 1: 데이터 요약 ---
print("-" * 80)
print("STEP 1: 데이터 요약 및 기초 통계")
print("-" * 80)

step1_prompt = f"""다음 고객 데이터의 기초 통계를 계산하세요:
{customer_data}

출력 항목:
1. 총 고객 수
2. 평균 연령
3. 평균 구매횟수
4. 평균 구매액
5. 최근 30일 이내 구매 고객 수

간단히 숫자만 나열하세요.
"""

result1 = await chain_agent.run(step1_prompt, model_settings=default_settings)
print("AI 응답:")
print(result1.output)

# --- STEP 2: 세그먼트 분류 (이전 대화 맥락 전달) ---
print("\n" + "-" * 80)
print("STEP 2: 고객 세그먼트 분류")
print("-" * 80)

step2_prompt = """앞서 분석한 데이터를 바탕으로 고객을 3개 세그먼트로 분류하세요:

분류 기준:
- VIP: 구매횟수 15회 이상 또는 총구매액 60만원 이상
- 일반: 구매횟수 5-14회 또는 총구매액 20-60만원
- 신규: 구매횟수 5회 미만 또는 총구매액 20만원 미만

각 세그먼트별로 해당 고객ID와 특징을 정리하세요.
"""

# message_history로 STEP 1의 맥락을 자동 전달
result2 = await chain_agent.run(
    step2_prompt,
    message_history=result1.new_messages(),
    model_settings=default_settings
)
print("AI 응답:")
print(result2.output)

# --- STEP 3: 액션 플랜 (누적된 대화 맥락 전달) ---
print("\n" + "-" * 80)
print("STEP 3: 세그먼트별 마케팅 전략")
print("-" * 80)

step3_prompt = """위 세그먼트 분석 결과를 바탕으로 각 그룹별 마케팅 전략을 제안하세요.

각 세그먼트별로:
1. 핵심 특징 (1문장)
2. 마케팅 목표
3. 구체적인 액션 2가지

전체 8문장 이내로 작성하세요.
"""

# STEP 1 + STEP 2의 대화 기록을 합쳐서 전달
all_messages = result1.new_messages() + result2.new_messages()
print(all_messages)
result3 = await chain_agent.run(
    step3_prompt,
    message_history=all_messages,
    model_settings=default_settings
)
print("AI 응답:")
print(result3.output)

In [ ]:
result2.new_messages()

## 프롬프트 엔지니어링 핵심 요약

### 5가지 핵심 원칙
1. **명확성**: 모호함 없이 구체적으로
2. **구조화**: 역할-작업-제약 구조 활용
3. **예시 제공**: Few-shot learning으로 일관성 확보
4. **단계별 사고**: 복잡한 문제는 Chain-of-Thought, 프롬프트 체이닝
5. **반복 개선**: 테스트하고 최적화하기

### PydanticAI 기본 구조
```python
# 역할 부여: Agent 생성 시 system_prompt
agent = Agent(
    model_id,
    system_prompt="당신은 ~입니다",
    output_type=MyOutputModel,  # 구조화된 출력 (선택)
)

# 실행: user prompt
result = await agent.run("실제 분석할 데이터")

# 체이닝: message_history로 맥락 전달
result2 = await agent.run("후속 질문", message_history=result.new_messages())
```

### 핵심 메시지
> "간단하고 명확하게" - 복잡한 프롬프트가 항상 좋은 것은 아님 \
> "반복 테스트" - 한 번에 완벽한 프롬프트는 없음 \
> "점진적 개선" - 작동하는 프롬프트를 조금씩 개선